# Lab 10: Hill Climbing Algorithms (2D Grid Search)

## Overview

In this lab, you will implement and compare multiple variants of the Hill Climbing algorithm on a 2D grid-based search space.

Each cell in the grid represents a numeric value (elevation or cost). The goal is to:

- Maximize values (find highest peak), OR  
- Minimize values (find lowest valley)

All algorithms operate on the SAME input grid to allow fair comparison.

---

## Algorithms Implemented

You will implement:

- Steepest Ascent Hill Climbing
- First Choice Hill Climbing
- Stochastic Hill Climbing
- Random Restart Hill Climbing
- Local Beam Search

---

## Input Format

Each input file contains:

### 1. Configuration
key=value format:
- mode = max|min
- grid_size = N
- start = row,col
- max_iterations = integer
- beam_width = integer (beam search only)
- restarts = integer (random restart only)

### 2. Grid

After:
grid=

Each line is a row of comma-separated integers.

---

## Output Format

Each algorithm produces a log file containing:

- Iteration number
- Current position
- Current value
- Move taken (U/D/L/R/None)
- Grid visualization with current position highlighted

---

## Important Rules

- No modification of OutputWriter
- No diagonal movement
- Only 4-directional movement allowed
- All algorithms must use the same input grid

## Step 1: Grid State Representation

Each state in the search space is represented as a position on a 2D grid.

A state consists of:
- row index
- column index
- reference to the grid

You will implement methods to:
- Retrieve the value of the current cell
- Generate valid neighboring states (up, down, left, right)
- Ensure boundary constraints are respected

This forms the foundation of all hill climbing variants.

In [38]:
class GridState:
    def __init__(self, row, col, grid, action="START"):
        self.row = row
        self.col = col
        self.grid = grid
        self.action = action

    def value(self):
        return self.grid[self.row][self.col]

    def get_neighbors(self):
        neighbors = []
        
        rows = len(self.grid)
        cols = len(self.grid[0])
        
        directions = [
            (-1, 0, "UP"),
            (1, 0, "DOWN"),
            (0, -1, "LEFT"),
            (0, 1, "RIGHT")
        ]
        
        for dr, dc, action in directions:
            new_row = self.row + dr
            new_col = self.col + dc
            
            if 0 <= new_row < rows and 0 <= new_col < cols:
                neighbors.append(GridState(new_row, new_col, self.grid, action))
        
        return neighbors

## Step 2: Run the Input Parser

Each algorithm reads its configuration from a file.

The file contains:
- mode (max/min)
- grid size
- start position
- optional parameters
- grid values

We must parse:
- configuration fields
- grid into a 2D integer matrix
- starting coordinates

In [39]:
class InputHandler:
    def __init__(self, file_path):
        self.file_path = file_path

    def load(self):
        config = {}
        grid = []
        parsing_grid = False

        with open(self.file_path, "r") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                if line.lower() == "grid=":
                    parsing_grid = True
                    continue

                if parsing_grid:
                    grid.append([int(x.strip()) for x in line.split(",")])
                else:
                    if "=" in line:
                        key, value = line.split("=", 1)
                        config[key.strip()] = value.strip()

        config["grid"] = grid

        r, c = config["start"].split(",")
        config["start_row"] = int(r.strip())
        config["start_col"] = int(c.strip())

        config["grid_size"] = int(config["grid_size"])

        config["max_iterations"] = int(config.get("max_iterations", 50))
        config["beam_width"] = int(config.get("beam_width", 3))
        config["restarts"] = int(config.get("restarts", 3))

        # validation
        if len(grid) == 0:
            raise ValueError("Grid missing")

        row_len = len(grid[0])
        for row in grid:
            if len(row) != row_len:
                raise ValueError("Grid not rectangular")

        return config

## Step 3: Run the Output Logger

Each iteration must be logged in a structured and readable format.

For every step, you will output:
- iteration number
- current position
- current value
- full grid visualization

The current position must be visually highlighted using a marker (e.g., X).

This allows clear understanding of how the algorithm moves through the search space.

In [40]:
class OutputWriter:
    def __init__(self, file_path):
        self.file_path = file_path

    def format_grid(self, grid, r, c):
        out = []
        for i in range(len(grid)):
            row = []
            for j in range(len(grid[i])):
                if i == r and j == c:
                    row.append(f"[{grid[i][j]}]")
                else:
                    row.append(str(grid[i][j]))
            out.append(" ".join(row))
        return "\n".join(out)

    def log_iteration(self, iteration, state):
        return (
            f"Iteration: {iteration}\n"
            f"Position: ({state.row},{state.col})\n"
            f"Value: {state.value()}\n"
            f"Move: {state.action}\n"
            f"{self.format_grid(state.grid, state.row, state.col)}"
        )

    def write(self, logs):
        with open(self.file_path, "w") as f:
            f.write("\n\n".join(logs))

## Step 4: Base Hill Climbing Logic

This class contains shared logic for all variants.

You will implement:
- comparison logic for max/min mode
- utility functions used by all algorithms

Do not implement full search logic here.

In [41]:
class HillClimbingBase:
    def __init__(self, config):
        self.grid = config["grid"]
        self.mode = config["mode"]

    def better(self, a, b):
        if self.mode == "max":
            return a > b
        else:
            return a < b

## Step 5: Hill Climbing Algorithms (Variants)

Implement each algorithm and make sure the logs print properly to the output file.

In [42]:
import random

class HillClimbing(HillClimbingBase):

    def steepest_ascent(self, start, writer):
        """
        STEEPEST ASCENT HILL CLIMBING

        Instructions:
        1. Start from the initial state.
        2. Generate ALL neighbors at each iteration.
        3. Select the best neighbor (based on mode: max/min).
        4. Move only if the best neighbor is better than current.
        5. Stop when no improvement is possible.
        """
        logs = []
        current = start
        iteration = 0

        while True:
            iteration += 1

            neighbors = current.get_neighbors()

            if not neighbors:
                logs.append(writer.log_iteration(iteration, current))
                break

            best = neighbors[0]
            for n in neighbors:
                if self.better(n.value(), best.value()):
                    best = n
            
            if self.better(best.value(), current.value()):
                current = best
            else:
                logs.append(writer.log_iteration(iteration, current))
                break

            logs.append(writer.log_iteration(iteration, current))

        return logs


    def first_choice(self, start, writer):
        """
        FIRST CHOICE HILL CLIMBING

        Instructions:
        1. Start from initial state.
        2. Check neighbors ONE BY ONE (do not evaluate all).
        3. Move immediately when a better neighbor is found.
        4. Do not search further after first improvement.
        5. Stop when no improving move exists.
        """
        logs = []
        current = start
        iteration = 0

        while True:
            iteration += 1
            moved = False

            neighbors = current.get_neighbors()

            if not neighbors:
                logs.append(writer.log_iteration(iteration, current))
                break

            for n in neighbors:
                if self.better(n.value(), current.value()):
                    current = n
                    moved = True
                    break
            

            logs.append(writer.log_iteration(iteration, current))

            if not moved:
                break

        return logs


    def stochastic(self, start, writer):
        """
        STOCHASTIC HILL CLIMBING

        Instructions:
        1. Generate all neighbors.
        2. Filter only better neighbors.
        3. Randomly choose ONE from better neighbors.
        4. If no better neighbor exists, stop.
        """
        logs = []
        current = start
        iteration = 0

        while True:
            iteration += 1

            neighbors = current.get_neighbors()

            if not neighbors:
                logs.append(writer.log_iteration(iteration, current))
                break

            better_neighbors = []
            for n in neighbors:
                if self.better(n.value(), current.value()):
                    better_neighbors.append(n)
            
            if better_neighbors:
                current = random.choice(better_neighbors)
            else:
                logs.append(writer.log_iteration(iteration, current))
                break

            logs.append(writer.log_iteration(iteration, current))

        return logs


    def random_restart(self, config, writer):
        """
        RANDOM RESTART HILL CLIMBING

        Instructions:
        1. Repeat hill climbing multiple times (restarts).
        2. Each restart starts from a RANDOM position.
        3. Run a full hill climbing process per restart.
        4. Keep track of the best overall solution.
        5. Log every iteration from every restart.
        """
        logs = []

        restarts = config["restarts"]

        rows = len(self.grid)
        cols = len(self.grid[0])

        best_overall = None

        for _ in range(restarts):

            r = random.randint(0, rows - 1)
            c = random.randint(0, cols - 1)

            current = GridState(r, c, self.grid)
            iteration = 0

            while True:
                iteration += 1

                neighbors = current.get_neighbors()

                if not neighbors:
                    logs.append(writer.log_iteration(iteration, current))
                    break

                best = neighbors[0]
                for n in neighbors:
                    if self.better(n.value(), best.value()):
                        best = n

                if self.better(best.value(), current.value()):
                    current = best
                else:
                    logs.append(writer.log_iteration(iteration, current))
                    break

                logs.append(writer.log_iteration(iteration, current))

            if best_overall is None or self.better(current.value(), best_overall.value()):
                best_overall = current


        return logs


    def beam_search(self, start, writer, beam_width):
        """
        LOCAL BEAM SEARCH

        Instructions:
        1. Maintain a list of 'beam_width' states.
        2. Expand ALL states in the beam.
        3. Collect all neighbors from beam states.
        4. Select top-k best states.
        5. Repeat until convergence or max iterations.
        """
        logs = []
        beam = [start]
        iteration = 0

        while True:
            iteration += 1

            all_neighbors = []

            for state in beam:
                all_neighbors.extend(state.get_neighbors())
            
            if not all_neighbors:
                logs.append(writer.log_iteration(iteration, beam[0]))
                break

            reverse = True if self.mode == "max" else False
            all_neighbors.sort(key=lambda x: x.value(), reverse=reverse)

            new_beam = all_neighbors[:beam_width]

            if all(not self.better(n.value(), beam[0].value()) for n in new_beam):
                logs.append(writer.log_iteration(iteration, beam[0]))
                break

            beam = new_beam

            logs.append(writer.log_iteration(iteration, beam[0]))

        return logs

## Step 10: Execution Pipeline

The main function handles:
- input file selection
- algorithm execution
- output file writing

Do NOT modify this method.

In [43]:
def run(input_file, output_file, method):
    parser = InputHandler(input_file)
    config = parser.load()

    solver = HillClimbing(config)
    writer = OutputWriter(output_file)

    start = GridState(
        config["start_row"],
        config["start_col"],
        config["grid"]
    )

    if method == "random_restart":
        logs = solver.random_restart(config, writer)
    elif method == "beam_search":
        logs = solver.beam_search(start, writer, config["beam_width"])
    else:
        logs = getattr(solver, method)(start, writer)

    writer.write(logs)


def main():
    input_file = "input.txt"

    jobs = [
        ("steepest_ascent", "out_steepest.txt"),
        ("first_choice", "out_first.txt"),
        ("stochastic", "out_stochastic.txt"),
        ("random_restart", "out_restart.txt"),
        ("beam_search", "out_beam.txt"),
    ]

    for method, out in jobs:
        run(input_file, out, method)


if __name__ == "__main__":
    main()